# Tugas - Klasifikasi dengan Support Vector Machine (SVM)

| Nama | NRP |
|------|-----|
| Catherine Aprilia Harlina | 5054251012 |

## Deskripsi Tugas
Pada tugas ini, kalian akan membangun model SVM Classifier untuk menyelesaikan sebuah masalah klasifikasi.

Dataset dapat diakses melalui link berikut:
[Link dataset akan diinformasikan terpisah]

Tujuan utama dari tugas ini adalah memahami cara kerja SVM secara praktik, khususnya:
1. Perbedaan hasil antara kernel Linear dan kernel RBF
2. Pengaruh parameter `C` dan `gamma` terhadap gejala overfitting/underfitting

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset dan Eksplorasi Awal
   - Memuat dataset, melihat struktur data, dan distribusi label.
2. Preprocessing
   - Memproses data agar siap digunakan dalam membangun model, termasuk feature scaling.
3. Eksperimen Model
   - Bangun model SVM dengan kernel Linear dan kernel RBF.
   - Lakukan eksperimen regularisasi dengan variasi nilai `C` dan `gamma`.
4. Evaluasi Model
   - Hitung metrik evaluasi dan visualisasikan hasilnya.
5. Analisis dan Kesimpulan
   - Bandingkan hasil eksperimen dan berikan kesimpulan.

# 1. Persiapan Dataset dan Eksplorasi Awal

In [ ]:
# Import library yang dibutuhkan (pandas, numpy, matplotlib, seaborn, sklearn, dll)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset dan tampilkan informasi dasar: jumlah baris dan kolom, tipe data, statistik deskriptif, dan jumlah missing value.
df = pd.read_csv('SVM/breast_cancer.csv')
print(f"Jumlah baris dan kolom: {df.shape}")
display(df.info())
display(df.describe().T[['mean', 'std', 'min', '50%', 'max']])
print("Jumlah missing value per kolom:\n", df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Tampilkan distribusi kelas target dalam bentuk bar plot.
# Perhatikan apakah dataset ini imbalanced.
plt.figure(figsize=(5, 3.5))
sns.countplot(data=df, x='diagnosis', palette='Set2')
plt.title('Distribusi Kelas Target (Diagnosis)')
plt.xlabel('Diagnosis (M = Malignant, B = Benign)')
plt.ylabel('Jumlah')
plt.show()

print("Persentase distribusi target (%):\n", df['diagnosis'].value_counts(normalize=True) * 100)

In [ ]:
# Tampilkan distribusi atau hubungan antar fitur (histogram, boxplot, atau pairplot), dibedakan berdasarkan kelas target.
# Ini membantu melihat apakah ada fitur yang punya batas pemisah yang jelas (linear) antar kelas, atau justru tumpang tindih.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x='diagnosis', y='radius_mean', ax=axes[0], palette='Set2')
axes[0].set_title('Radius Mean vs Diagnosis')

sns.boxplot(data=df, x='diagnosis', y='texture_mean', ax=axes[1], palette='Set2')
axes[1].set_title('Texture Mean vs Diagnosis')

plt.tight_layout()
plt.show()

## Lakukan Exploratory Data Analysis (EDA) tambahan yang diperlukan.

# 2. Preprocessing

In [ ]:
# Tangani missing value (jika ada) dengan strategi yang sesuai.
# Pastikan strategi yang dipilih tidak memperkenalkan data leakage.
cols_to_drop = [c for c in ['id', 'Unnamed: 32'] if c in df.columns]
df = df.drop(columns=cols_to_drop)
df = df.dropna(axis=1, how='all')

In [ ]:
# Jika terdapat fitur kategorikal, lakukan encoding yang sesuai (misalnya One-Hot Encoding atau Ordinal Encoding).
if df['diagnosis'].dtype == 'object':
    df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

In [ ]:
# Pisahkan fitur (X) dan target (y).
# Bagi data menjadi train set dan test set (misalnya 80:20).
# Gunakan stratify=y agar distribusi kelas tetap proporsional.
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Catatan penting:** SVM adalah algoritma berbasis jarak (margin dihitung dari posisi antar titik data), sehingga sangat sensitif terhadap skala fitur. Fitur dengan rentang nilai besar bisa mendominasi perhitungan jarak meskipun sebenarnya tidak lebih penting. Lakukan **feature scaling** (misalnya `StandardScaler`) pada fitur numerik sebelum training. Ingat: `fit` scaler hanya pada train set, lalu `transform` ke train dan test set, agar tidak terjadi data leakage.

In [ ]:
# Lakukan feature scaling (StandardScaler) pada fitur numerik.
# fit_transform pada X_train, transform saja pada X_test.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Eksperimen Model

## 3.1 SVM dengan Kernel Linear

Kernel Linear mencari hyperplane pemisah berupa garis/bidang lurus, tanpa proyeksi ke dimensi yang lebih tinggi. Cocok digunakan sebagai baseline, terutama jika dari hasil EDA data terlihat cukup terpisah secara linear.

In [ ]:
# Inisialisasi dan latih model SVC dengan kernel='linear' menggunakan train set (yang sudah discaling).
model_linear = SVC(kernel='linear', random_state=42)
model_linear.fit(X_train_scaled, y_train)

In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label untuk keperluan evaluasi.
y_pred_linear = model_linear.predict(X_test_scaled)

## 3.2 SVM dengan Kernel RBF

Kernel RBF (Radial Basis Function) memproyeksikan data ke ruang berdimensi lebih tinggi melalui kernel trick, sehingga mampu menangani batas keputusan yang non-linear.

In [ ]:
# Inisialisasi dan latih model SVC dengan kernel='rbf' menggunakan train set.
# Gunakan nilai hyperparameter lain (C) yang sama seperti model Linear di atas, agar perbandingan adil.
model_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
model_rbf.fit(X_train_scaled, y_train)

In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label untuk keperluan evaluasi.
y_pred_rbf = model_rbf.predict(X_test_scaled)

## 3.3 Eksperimen Regularisasi: Pengaruh C dan Gamma terhadap Overfitting

Parameter `C` mengatur trade-off antara margin lebar dan toleransi kesalahan klasifikasi, sedangkan `gamma` (khusus kernel RBF) mengatur seberapa lokal pengaruh tiap titik data. Pada bagian ini kalian akan mengamati bagaimana perubahan nilai `C` (dan/atau `gamma`) memengaruhi akurasi training dan testing.

In [ ]:
# Latih beberapa model SVC (kernel RBF) dengan variasi nilai C,
# misalnya 0.01, 0.1, 1, 10, 100.
# Gunakan gamma='scale' atau nilai gamma tetap agar perbandingan fokus pada pengaruh C.
c_values = [0.01, 0.1, 1, 10, 100]
models = [SVC(kernel='rbf', C=c, gamma='scale', random_state=42).fit(X_train_scaled, y_train) for c in c_values]

In [ ]:
# Hitung akurasi pada train set dan test set untuk setiap nilai C yang dicoba.
# Simpan hasilnya dalam list atau dictionary agar mudah diplot.
train_accs = [accuracy_score(y_train, m.predict(X_train_scaled)) for m in models]
test_accs = [accuracy_score(y_test, m.predict(X_test_scaled)) for m in models]

In [ ]:
# Plot grafik akurasi training vs testing terhadap nilai C (gunakan sumbu x logaritmik) dalam satu chart.
# Amati pada nilai C berapa model mulai menunjukkan gejala overfitting
# (akurasi training terus naik, sementara akurasi testing mulai stagnan atau menurun).
plt.figure(figsize=(6, 4))
plt.plot(c_values, train_accs, marker='o', label='Train Accuracy')
plt.plot(c_values, test_accs, marker='s', label='Test Accuracy')
plt.xscale('log')
plt.xlabel('Nilai C (Log Scale)')
plt.ylabel('Akurasi')
plt.title('Akurasi Training vs Testing terhadap Nilai C')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 4. Evaluasi Model

In [ ]:
# Tampilkan metrik evaluasi (Accuracy, Precision, Recall, F1-Score) untuk model kernel Linear dan RBF.
# Sajikan dalam satu tabel perbandingan agar mudah dibandingkan.
def eval_metrics(y_true, y_pred, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred)
    }

df_eval = pd.DataFrame([
    eval_metrics(y_test, y_pred_linear, 'Kernel Linear'),
    eval_metrics(y_test, y_pred_rbf, 'Kernel RBF (C=1.0)')
])
display(df_eval)

In [ ]:
# Tampilkan Confusion Matrix untuk model kernel Linear dan RBF dalam bentuk heatmap.
# Susun keduanya dalam satu baris subplot.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

sns.heatmap(confusion_matrix(y_test, y_pred_linear), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix - Linear')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_rbf), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Confusion Matrix - RBF')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan jumlah support vectors dari model dengan performa terbaik (atribut model.support_vectors_ atau model.n_support_).
# Bandingkan jumlah support vectors antara model Linear dan RBF.
print(f"Support Vectors - Linear : {model_linear.n_support_.sum()} (Malignant: {model_linear.n_support_[1]}, Benign: {model_linear.n_support_[0]})")
print(f"Support Vectors - RBF    : {model_rbf.n_support_.sum()} (Malignant: {model_rbf.n_support_[1]}, Benign: {model_rbf.n_support_[0]})")

In [ ]:
# Jika memungkinkan, pilih dua fitur (atau reduksi dimensi dengan PCA ke 2D) lalu visualisasikan decision boundary
# beserta margin dan support vectors dari model terbaik.
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)

svm_pca = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_train_pca, y_train)

x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 4.5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap='coolwarm', edgecolors='k', s=20)
plt.scatter(svm_pca.support_vectors_[:, 0], svm_pca.support_vectors_[:, 1], 
            s=70, facecolors='none', edgecolors='k', linewidths=1.2, label='Support Vectors')
plt.title('Decision Boundary & Support Vectors (PCA 2D - RBF)')
plt.legend()
plt.show()

# 5. Analisis

### Jelaskan secara detail hasil analisis yang didapatkan dari eksperimen yang telah dilakukan.

Beberapa hal yang perlu dibahas:
- **Bandingkan performa SVM dengan kernel Linear vs RBF**: Kedua kernel memberikan hasil yang seimbang dan sangat tinggi pada dataset Breast Cancer. Hal ini mengindikasikan bahwa fitur-fitur pada dataset ini pada dasarnya sudah terpisah cukup baik secara linear di ruang fitur aslinya.
- **Hasil eksperimen regularisasi (nilai C)**: Pada nilai $C$ sangat kecil ($C=0.01$), model mengalami *underfitting* karena batas margin terlalu toleran. Performa testing optimal dicapai pada kisaran $C=1.0$ hingga $C=10$. Ketika $C$ dinaikkan ke $100$, akurasi training terus meningkat mendekati $100\%$, menunjukkan tanda awal *overfitting* di mana batas keputusan menjadi terlalu selektif terhadap data training.
- **Perbandingan jumlah support vectors**: Model Linear menggunakan support vector yang lebih sedikit dibandingkan RBF. Makin sedikit jumlah support vector, makin sederhana hyperplane yang dibentuk, yang berarti model lebih efisien dalam komputasi dan tidak rentan terhadap noise.

# 6. Kesimpulan dan Saran

### Berikan kesimpulan dari eksperimen yang telah dilakukan.

**Kesimpulan:**
- Feature scaling (`StandardScaler`) adalah tahap wajib untuk SVM agar fitur dengan rentang nilai besar tidak mendominasi perhitungan jarak margin.
- Kernel RBF dengan konfigurasi $C=1.0$ dan `gamma='scale'` terbukti efisien dan mampu menghasilkan performa generalisasi yang stabil tanpa overfitting berlebih.

**Saran:**
- Pencarian kombinasi hiperparameter dapat diotomatisasi dengan `GridSearchCV` untuk menguji variasi $C$ dan $\gamma$ secara serentak.
- Bisa dicoba eksperimen tambahan menggunakan kernel Polynomial atau metode reduksi dimensi lain untuk melihat pengaruhnya terhadap batas keputusan.